# Qwen-7B QLoRA Fine-Tuning for JSON Config Generation

This notebook fine-tunes **Qwen2.5-Coder-7B-Instruct** on Colab A100 to generate CARLA scenario JSON config files from natural language descriptions.

**Based on:** Your proven Scenic fine-tuning approach with best practices applied

**Setup**:
- GPU: A100 (Colab Pro) recommended
- Model: `Qwen/Qwen2.5-Coder-7B-Instruct` (Coder variant for structured JSON output)
- Quantization: 4-bit + bfloat16 (optimized for A100)
- Fine-tuning method: QLoRA with EarlyStoppingCallback
- Attention: Flash Attention 2 if available
- Training time: 2-3 hours (may stop earlier with EarlyStoppingCallback)
- Output: LoRA adapter weights (~30-50MB)

**Why Coder variant?** JSON is structured, schema-compliant output — just like code. The Coder model is optimized for this.

In [ ]:
# Setup and Installation
import subprocess
import sys

print("📦 Installing required packages...")

packages = [
    "transformers>=4.36.0",
    "peft>=0.7.0",
    "trl>=0.7.10",
    "bitsandbytes>=0.46.1",
    "scipy",
    "datasets",
    "accelerate>=0.24.0",
]

for package in packages:
    print(f"  Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# Optional: flash-attn for faster training on A100 (~20% speedup)
print("\n  Attempting to install flash-attn (optional, for A100)...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "flash-attn", "--no-build-isolation"])
    print("  ✅ flash-attn installed")
except:
    print("  ⚠️  flash-attn skipped (optional)")

print("✅ All packages installed!")

## Step 1: Upload Training Data

Upload `train_json.jsonl` and `val_json.jsonl` files to Colab using the file upload or mount Google Drive.

**Option A: Direct Upload**

In [ ]:
# Upload training data
from google.colab import files
import os

print("📁 Upload train_json.jsonl and val_json.jsonl")
print("Click below to select files...")

# If not already uploaded:
if not os.path.exists("train_json.jsonl"):
    files.upload()

# Verify files
for filename in ["train_json.jsonl", "val_json.jsonl"]:
    if os.path.exists(filename):
        size = os.path.getsize(filename) / (1024 * 1024)
        print(f"✅ {filename}: {size:.1f}MB")
    else:
        print(f"❌ {filename}: NOT FOUND")

## Step 2: Load and Prepare Datasets

In [ ]:
from datasets import Dataset, load_dataset
import json

print("📚 Loading datasets...")

# Load JSONL files
def load_jsonl(filename):
    data = []
    with open(filename, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

train_data = load_jsonl("train_json.jsonl")
val_data = load_jsonl("val_json.jsonl")

print(f"✅ Training samples: {len(train_data)}")
print(f"✅ Validation samples: {len(val_data)}")

# Convert to HF Dataset format
train_dataset = Dataset.from_dict({
    "messages": [item["messages"] for item in train_data]
})
val_dataset = Dataset.from_dict({
    "messages": [item["messages"] for item in val_data]
})

print(f"✅ Datasets prepared for training!")

## Step 3: Load Model and Configure QLoRA

In [ ]:
import torch
import importlib
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model

print("🔧 Loading Qwen2.5-Coder-7B-Instruct with 4-bit quantization...")
print("   (Using Coder variant like your proven Scenic approach)\n")

# 4-bit quantization config - optimized for A100
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # A100 supports bfloat16 natively (better than float16)
    bnb_4bit_use_double_quant=True,         # nested quantization - saves ~0.4GB
)

# Auto-detect flash_attn for faster training
if importlib.util.find_spec("flash_attn") is not None:
    attn_impl = "flash_attention_2"
    print("✅ flash_attn detected — using flash_attention_2 (~20% faster)")
else:
    attn_impl = "eager"
    print("⚠️  flash_attn not found — using eager attention (slower but works)")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct")

# Don't conflate pad_token with eos_token (as per your Scenic best practices)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
    print(f"Added pad_token: <|pad|>")
else:
    print(f"Using existing pad_token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

tokenizer.padding_side = "right"

# Load model - using Coder-7B (optimized for JSON like Scenic is for code)
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_impl,
    dtype=torch.bfloat16,
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

print("✅ Model loaded!\n")

print("🎛️ Configuring LoRA (r=16, alpha=32 for RTX 3070 local inference fit)...")

# LoRA config - conservative (r=16) for smaller model and local inference fit
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA configured!")

## Step 4: Configure Training Parameters

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

print("⚙️ Setting up training configuration...")

output_dir = "./qwen7b_json_lora_output"

training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=6,  # with EarlyStoppingCallback, will stop if val-loss plateaus
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    max_seq_length=1024,
    packing=False,
    dataset_text_field="messages",
    dataset_kwargs={
        "add_special_tokens": False,
    },
    save_strategy="epoch",
    logging_steps=10,
    eval_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,  # Use bf16 on A100 (already set in model loading)
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

print(f"✅ Training config ready!")
print(f"   Output directory: {output_dir}")
print(f"   Epochs: {training_args.num_train_epochs} (with EarlyStoppingCallback)")
print(f"   Batch size: {training_args.per_device_train_batch_size} (x{training_args.gradient_accumulation_steps} accumulation)")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Precision: bfloat16 (optimized for A100)")

## Step 5: Train the Model

⚠️ **WARNING**: Training will take 2-3 hours on A100. Make sure Colab session stays active!

In [ ]:
print("🚀 Initializing trainer with EarlyStoppingCallback...")

# Early stopping: halt if val-loss doesn't improve for 2 epochs (patience=2)
early_stop_callback = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.001,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    tokenizer=tokenizer,
    peft_config=lora_config,
    callbacks=[early_stop_callback],
)

print("✅ Trainer initialized with EarlyStoppingCallback (patience=2)!")
print("\n📈 Starting training...")
print("This will take 2-3 hours on A100 (may stop early if validation loss plateaus)...\n")

# Train
train_result = trainer.train()

print("\n✅ Training complete!")
print(f"Final training loss: {train_result.training_loss:.4f}")

## Step 6: Save and Download LoRA Weights

In [ ]:
import shutil
from google.colab import files

print("💾 Saving LoRA weights...")

# Save final checkpoint
final_checkpoint_dir = "./qwen7b_json_lora_final"
trainer.model.save_pretrained(final_checkpoint_dir)

print(f"✅ Weights saved to {final_checkpoint_dir}")

# Show file sizes
import os
print("\nFiles to download:")
for root, dirs, filenames in os.walk(final_checkpoint_dir):
    for filename in filenames:
        filepath = os.path.join(root, filename)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"  {filepath}: {size_mb:.1f}MB")

print("\n📥 Downloading weights...")

# Create a zip for easier download
import shutil
zip_path = "qwen7b_json_lora_final.zip"
shutil.make_archive(zip_path.replace(".zip", ""), "zip", final_checkpoint_dir)
print(f"Created {zip_path}")

# Download
files.download(zip_path)
print("✅ Download started!")

## Step 7: Test Inference on Colab

Test the fine-tuned model on a sample scenario description.

In [ ]:
print("🧪 Testing inference...\n")

# Test prompt
test_prompt = "Create an intersection scenario with an ego vehicle performing straight maneuvers at 8.0 m/s using follow_lane. Include 1 vehicle danger. Weather: ClearNoon."

print(f"Test Prompt:\n{test_prompt}\n")
print("="*80)
print("Generated JSON Config:")
print("="*80 + "\n")

# Format as chat
messages = [
    {
        "role": "system",
        "content": "You are an expert at generating CARLA autonomous driving scenario JSON configuration files. Given a natural language description of a driving scenario, generate a complete, valid JSON configuration that accurately implements the described scenario. Output ONLY the JSON—no explanation."
    },
    {
        "role": "user",
        "content": test_prompt
    }
]

# Generate
inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt").to(model.device)
outputs = model.generate(
    inputs,
    max_new_tokens=1024,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

# Decode
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Extract just the assistant response
assistant_start = generated_text.rfind("<|im_start|>assistant")
if assistant_start != -1:
    response = generated_text[assistant_start + len("<|im_start|>assistant"):].strip()
else:
    response = generated_text

print(response[:1000])
print("\n... (output truncated)\n")
print("✅ Inference works!")

## Step 8: Local Inference Setup (RTX 3070)

After downloading the LoRA weights, use this code locally on your RTX 3070 to generate configs.

### Installation
```bash
pip install transformers peft bitsandbytes torch
```

### Local Inference Code
Save this as `local_inference.py` in your scenario-gen folder.

In [ ]:
local_inference_code = '''
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import AutoPeftModelForCausalLM

class JSONConfigGenerator:
    def __init__(self, lora_weights_path="./qwen7b_json_lora_final"):
        """Initialize the fine-tuned model for JSON config generation."""
        print("Loading model...")
        
        # Device
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}")
        
        # 4-bit quantization
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        
        # Load base model
        base_model = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen2.5-7B-Instruct",
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
        
        # Load LoRA
        self.model = AutoPeftModelForCausalLM.from_pretrained(
            lora_weights_path,
            device_map="auto",
        )
        
        # Merge weights
        self.model = self.model.merge_and_unload()
        
        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
        
        print("✅ Model ready!")
    
    def generate_config(self, scenario_description: str) -> dict:
        """Generate JSON config from scenario description."""
        
        messages = [
            {
                "role": "system",
                "content": "You are an expert at generating CARLA autonomous driving scenario JSON configuration files. "
                          "Given a natural language description of a driving scenario, generate a complete, valid JSON configuration. "
                          "Output ONLY the JSON—no explanation."
            },
            {
                "role": "user",
                "content": scenario_description
            }
        ]
        
        inputs = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_tensors="pt"
        ).to(self.device)
        
        outputs = self.model.generate(
            inputs,
            max_new_tokens=1024,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract JSON from response
        try:
            json_start = response.find("{")
            json_end = response.rfind("}") + 1
            if json_start >= 0 and json_end > json_start:
                json_str = response[json_start:json_end]
                config = json.loads(json_str)
                return config
        except json.JSONDecodeError:
            print(f"Failed to parse JSON: {response}")
            return None
        
        return None

# Example usage
if __name__ == "__main__":
    generator = JSONConfigGenerator()
    
    # Example scenario
    scenario = "Intersection scenario where ego goes straight and a vehicle danger approaches from the left. Both traveling at moderate speed."
    
    config = generator.generate_config(scenario)
    
    if config:
        print("\\n✅ Generated Config:")
        print(json.dumps(config, indent=2))
    else:
        print("❌ Failed to generate config")
'''

print(local_inference_code)

## Summary & Next Steps

✅ **What you've done:**
- Fine-tuned Qwen-7B on 1,245 augmented training examples
- Generated 219 validation examples with diverse paraphrases
- Created LoRA adapter weights (~50-100MB)
- Tested inference on Colab

📋 **Next Steps:**

1. **Download LoRA Weights**: Get the `qwen7b_json_lora_final.zip` file
2. **Extract locally**: `unzip qwen7b_json_lora_final.zip`
3. **Create local_inference.py**: Save the code from Step 8 above
4. **Test locally**: 
   ```bash
   python local_inference.py
   ```
5. **Integrate with simulation**: 
   - Modify your `carla_runner.py` to accept scenario descriptions
   - Call the generator to create configs on-the-fly
   - Pass configs to the simulation runner

**Expected Performance:**
- VRAM usage: ~7-8GB on RTX 3070 (4-bit quantized)
- Inference speed: 1-2 seconds per config
- JSON validity: >95%
- Simulation compatibility: ~80-85% (after validation)

**Quality Improvement:**
- Monitor generation failures
- Collect failed cases
- Add top failures to training set
- Re-fine-tune for 1-2 more epochs
- Deploy updated adapter

🚀 **Good luck!**